Progetto Machine Learning

connetto colab al mio drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Setup del Codice (Modalità Condivisa / Salvataggio su Drive)
Invece di scaricare il codice ogni volta nella memoria temporanea, lo cloniamo **una sola volta** sul Google Drive. In questo modo le modifiche che fate al codice Python rimangono salvate per sempre.

In [ ]:
# Definiamo il percorso dove vogliamo tenere il codice sul Drive (modifica se necessario)
project_dir = '/content/drive/MyDrive/Progetto_Machine_Learning'#'/content/drive/.shortcut-targets-by-id/1sNCZabQyeQ6JKeZVdieFE3Jy9JBtb7U6/Progetto_Machine_Learning'
repo_dir = f'{project_dir}/Visual-Place-Recognition-Project'
%cd {repo_dir}


/content/drive/MyDrive/Progetto_Machine_Learning/Visual-Place-Recognition-Project


In [ ]:
%cd /content/drive/MyDrive/Progetto_Machine_Learning/Visual-Place-Recognition-Project/image-matching-models
import os, subprocess, sys

CACHE_DIR = f"{project_dir}/pip_cache"
MODEL_DIR = f"{repo_dir}/image-matching-models"

os.makedirs(CACHE_DIR, exist_ok=True)

def pip_show(package):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", package],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    return result.returncode == 0

# Installa faiss solo se non è già installato nel runtime corrente
if not pip_show("faiss-cpu"):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--cache-dir", CACHE_DIR,
        "faiss-cpu"
    ])
else:
    print("faiss-cpu già installato")

# Installa image-matching-models solo se non è già installato
if not pip_show("image-matching-models"):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--cache-dir", CACHE_DIR,
        "-e", MODEL_DIR
    ])
else:
    print("image-matching-models già installato")

/content/drive/MyDrive/Progetto_Machine_Learning/Visual-Place-Recognition-Project/image-matching-models


# -------------------------------------------------------------------
# VERSIONE ALTERNATIVA / FALLBACK
# Usare questa cella SOLO se la versione leggera con:
#     pip install -e MODEL_DIR
# non funziona e dà errori tipo ModuleNotFoundError.
#
# Questa versione installa image-matching-models con [all],
# quindi include tutte le dipendenze extra richieste dal progetto.
# È più lenta, ma più completa.
# -------------------------------------------------------------------

In [ ]:


import os
import subprocess
import sys
from pathlib import Path

# Cartella su Google Drive dove salviamo la cache dei pacchetti pip.
# In questo modo, dopo la prima esecuzione, Colab può riutilizzare i file già scaricati.
CACHE_DIR = Path("/content/drive/MyDrive/Progetto_Machine_Learning/pip_cache")

# Cartella principale del progetto su Google Drive.
PROJECT_DIR = Path("/content/drive/MyDrive/Progetto_Machine_Learning/Visual-Place-Recognition-Project")

# Cartella del pacchetto image-matching-models.
MODEL_DIR = PROJECT_DIR / "image-matching-models"

# Creiamo la cache se non esiste.
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Controllo che la cartella del modello esista.
if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Non trovo la cartella del modello: {MODEL_DIR}")

def pip_show(package_name):
    """
    Controlla se un pacchetto è già installato nel runtime corrente di Colab.
    Attenzione: quando il runtime viene resettato, i pacchetti vanno reinstallati.
    """
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", package_name],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    return result.returncode == 0

# ------------------------------------------------------------
# 1. Installazione di faiss-cpu
# ------------------------------------------------------------

if not pip_show("faiss-cpu"):
    print("Installo faiss-cpu...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--cache-dir", str(CACHE_DIR),
        "faiss-cpu"
    ])
else:
    print("faiss-cpu già installato nel runtime corrente.")

# ------------------------------------------------------------
# 2. Installazione completa di image-matching-models con [all]
# ------------------------------------------------------------
# Questa è la parte importante:
#     -e str(MODEL_DIR) + "[all]"
#
# Significa: installa il pacchetto locale in modalità editable
# e includi tutte le dipendenze extra definite in [all].
# È più lenta della versione senza [all], ma più sicura.

if not pip_show("image-matching-models"):
    print("Installo image-matching-models con tutte le dipendenze extra [all]...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--cache-dir", str(CACHE_DIR),
        "-e", str(MODEL_DIR) + "[all]"
    ])
else:
    print("image-matching-models già installato nel runtime corrente.")

print("Installazione completata.")

faiss-cpu già installato nel runtime corrente.
image-matching-models già installato nel runtime corrente.
Installazione completata.


In [ ]:
!pip install faiss-cpu
%cd ..

/content/drive/.shortcut-targets-by-id/1sNCZabQyeQ6JKeZVdieFE3Jy9JBtb7U6/Progetto_Machine_Learning


### Estrazione Dataset (Memoria Temporanea per Massime Prestazioni)
I file `.zip` pesanti sono conservati al sicuro sul Drive. Tuttavia, per non creare colli di bottiglia e rallentare la GPU, **scompattiamo i file nella memoria SSD temporanea di Colab (`/content/datasets/`)**. Questo garantisce una lettura delle immagini alla massima velocità.

In [ ]:
!mkdir -p /content/datasets
import os
import shutil
import zipfile
from pathlib import Path

# Cartella dove hai salvato gli zip su Google Drive
DATASET_ZIP_DIR = Path("/content/drive/MyDrive/Progetto_Machine_Learning/datasets")

# Cartella veloce temporanea di Colab dove estraiamo i dataset
DATASET_DIR = Path("/content/datasets")

# Cartella temporanea locale di Colab dove copiamo prima gli zip
LOCAL_ZIP_DIR = Path("/content/local_zips")

DATASET_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ZIP_DIR.mkdir(parents=True, exist_ok=True)

datasets = [
    "tokyo_xs.zip",
    "sf_xs.zip",
    "svox.zip",
    "gsv_xs.zip"
]

for zip_name in datasets:
    marker = DATASET_DIR / f".{zip_name}.done"

    if marker.exists():
        print(f"{zip_name} già estratto in questo runtime, salto.")
        continue

    drive_zip = DATASET_ZIP_DIR / zip_name
    local_zip = LOCAL_ZIP_DIR / zip_name

    if not drive_zip.exists():
        raise FileNotFoundError(f"Non trovo il file: {drive_zip}")

    print(f"Copio {zip_name} da Drive a /content...")
    shutil.copy2(drive_zip, local_zip)

    print(f"Estraggo {zip_name} in {DATASET_DIR}...")
    with zipfile.ZipFile(local_zip, "r") as z:
        z.extractall(DATASET_DIR)

    marker.touch()
    local_zip.unlink()

    print(f"{zip_name} completato.")

print("Estrazione completata.")

Copio svox.zip da Drive a /content...
Estraggo svox.zip in /content/datasets...
svox.zip completato.
Estrazione completata.


valutazione


In [ ]:
%cd /content/drive/.shortcut-targets-by-id/1sNCZabQyeQ6JKeZVdieFE3Jy9JBtb7U6/Progetto_Machine_Learning/Visual-Place-Recognition-Project

/content/drive/.shortcut-targets-by-id/1sNCZabQyeQ6JKeZVdieFE3Jy9JBtb7U6/Progetto_Machine_Learning/Visual-Place-Recognition-Project


In [ ]:
%cd /content/drive/MyDrive/Progetto_Machine_Learning

/content/drive/MyDrive/Progetto_Machine_Learning


retrieval


In [ ]:
!python VPR-methods-evaluation/main.py \
--num_workers 8 \
--batch_size 16 \
--log_dir "/content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/netvlad_vgg16_4096/svox_sun/default_metric" \
--method="netvlad" \
--backbone="VGG16" \
--descriptors_dimension=4096 \
--distance_metric=l2 \
--image_size 512 512 \
--database_folder '/content/datasets/svox/images/test/gallery' \
--queries_folder '/content/datasets/svox/images/test/queries_sun' \
--num_preds_to_save 20 \
--recall_values 1 5 10 20 \
--save_for_uncertainty

2026-05-15 15:46:30 VPR-methods-evaluation/main.py --num_workers 8 --batch_size 16 --log_dir /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/netvlad_vgg16_4096/svox_sun/default_metric --method=netvlad --backbone=VGG16 --descriptors_dimension=4096 --distance_metric=l2 --image_size 512 512 --database_folder /content/datasets/svox/images/test/gallery --queries_folder /content/datasets/svox/images/test/queries_sun --num_preds_to_save 20 --recall_values 1 5 10 20 --save_for_uncertainty
2026-05-15 15:46:30 Arguments: Namespace(positive_dist_threshold=25, method='netvlad', backbone='VGG16', descriptors_dimension=4096, database_folder='/content/datasets/svox/images/test/gallery', queries_folder='/content/datasets/svox/images/test/queries_sun', num_workers=8, batch_size=16, log_dir='/content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/netvlad_vgg16_4096/svox_sun/default_metric', device='cuda', recall_values=[1, 5, 10, 20], no_labels=False, num_preds_to_save=20, save_o

image matching

In [ ]:
!python match_queries_preds.py \
--preds-dir '/content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/cosplace_resnet18_512/svox_sun/default_metric/2026-05-15_15-37-26/preds' \
--matcher 'loftr' \
--device 'cuda' \
--num-preds 20

/content/drive/MyDrive/Progetto_Machine_Learning/Visual-Place-Recognition-Project/image-matching-models/matching/third_party/LightGlue/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
100% 854/854 [59:44<00:00,  4.20s/it]

--- TIMING SUMMARY ---
Total Inference Time: 3303.764538 s
Mean Inference Time per Query: 3.868577 s/query
✅ Local stats saved to: /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/cosplace_resnet18_512/svox_sun/default_metric/2026-05-15_15-37-26/preds_loftr/matching_stats.torch
✅ Global benchmark updated in: benchmark_matching_times.csv



reranking



In [ ]:
!python reranking.py \
--preds-dir '/content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/netvlad_vgg16_4096/svox_sun/default_metric/2026-05-15_15-46-30/preds' \
--inliers-dir '/content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/netvlad_vgg16_4096/svox_sun/default_metric/2026-05-15_15-46-30/preds_loftr' \
--num-preds 20 \
--recall-values 1 5 10 20

100% 854/854 [11:26<00:00,  1.24it/s]
R@1: 64.6, R@5: 66.9, R@10: 67.8, R@20: 69.0


analisi file z_data post retrieval


In [ ]:
import torch
import numpy as np

# Inserisci qui il percorso esatto del file che vedi nello screenshot
path = '/content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/cosplace_resnet18_512/svox_sun/default_metric/2026-05-15_15-37-26/z_data.torch'
# Caricamento
data = torch.load(path,weights_only=False)

print("--- PERFORMANCE DEL MODELLO VPR ---")
# Stampiamo i tempi totali (se esistono nel file)
if 'time_extraction' in data and 'time_retrieval' in data:
    num_queries_totali = data['predictions'].shape[0]
    print("Number of queries totalied is: ", num_queries_totali)
    t_ext = data['time_extraction']
    t_ret = data['time_retrieval']
    t_tot = t_ext + t_ret

    print(f"Tempo totale di estrazione (Query): {t_ext:.4f} s")
    print(f"Tempo totale di retrieval (FAISS):  {t_ret:.4f} s")
    print(f"Tempo complessivo:                  {t_tot:.4f} s")
    print(f"Tempo medio per singola query:      {(t_tot / num_queries_totali):.4f} s/query")
else:
    print("ATTENZIONE: Tempi non trovati nel file z_data.torch (forse è stato generato col vecchio script?)")

print("\n--- ISPEZIONE DELLE PRIME QUERY ---")
num_queries_da_vedere = 5

for q in range(num_queries_da_vedere):
    print(f"\n=== Analisi Query {q} ===")

    # Stampiamo solo le prime 5 predizioni per comodità di lettura
    print(f"Predizioni (top 5): {data['predictions'][q][:5]}")

    # Stampiamo le distanze associate a quelle prime 5 predizioni
    print(f"Distanze (top 5):   {data['distances'][q][:5]}")

    # Stampiamo i positivi (se la lista è molto lunga, ne mostriamo solo alcuni)
    positivi = data['positives_per_query'][q]
    print(f"Positivi (GT):      {positivi[:10]} ... (totale: {len(positivi)})")

--- PERFORMANCE DEL MODELLO VPR ---
Number of queries totalied is:  854
Tempo totale di estrazione (Query): 11.6190 s
Tempo totale di retrieval (FAISS):  0.2750 s
Tempo complessivo:                  11.8940 s
Tempo medio per singola query:      0.0139 s/query

--- ISPEZIONE DELLE PRIME QUERY ---

=== Analisi Query 0 ===
Predizioni (top 5): [2218 2198 2229 2248 2266]
Distanze (top 5):   [1.3336617 1.3386672 1.3526946 1.3723986 1.3773236]
Positivi (GT):      [1995 1971 1943 1982 1983 1970 1981 1980 1950 1942] ... (totale: 22)

=== Analisi Query 1 ===
Predizioni (top 5): [1981 2170 2218 2229 2248]
Distanze (top 5):   [1.3726285 1.4022254 1.4093665 1.4147158 1.4212416]
Positivi (GT):      [1995 1971 1943 1982 1983 1970 1981 1980 1950 1942] ... (totale: 22)

=== Analisi Query 2 ===
Predizioni (top 5): [1981 1969 2655 1995 2266]
Distanze (top 5):   [1.0369322 1.1952759 1.3646871 1.3806312 1.421552 ]
Positivi (GT):      [1995 1971 1943 1982 1983 1970 1981 1980 1950 1942] ... (totale: 28)

===

analisi file xxx.torch singole query post image matching

In [ ]:
import torch
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

def esplora_lista_torch(percorso_file):
    print(f"🔍 Ispezionando il file: {percorso_file.name}")
    print("-" * 50)

    try:
        dati = torch.load(percorso_file, map_location='cpu', weights_only=False)

        if isinstance(dati, list):
            print(f"📦 Tipo di contenitore: LISTA con {len(dati)} elementi.\n")

            # Estraiamo il primo elemento (probabilmente il match con la Top-1)
            primo_elemento = dati[0]
            print(f"Il primo elemento (Indice 0) è di tipo: {type(primo_elemento).__name__}\n")

            if isinstance(primo_elemento, dict):
                print(f"Trovato il Dizionario! Contiene {len(primo_elemento.keys())} chiavi:")
                for chiave, valore in primo_elemento.items():
                    if isinstance(valore, (torch.Tensor, np.ndarray)):
                        print(f"  🔑 {chiave:<15} | Tipo: {type(valore).__name__:<13} | Shape: {list(valore.shape)}")
                    elif isinstance(valore, (list, tuple)):
                        print(f"  🔑 {chiave:<15} | Tipo: {type(valore).__name__:<13} | Lunghezza: {len(valore)}")
                    else:
                        print(f"  🔑 {chiave:<15} | Tipo: {type(valore).__name__:<13} | Valore: {valore}")

            elif isinstance(primo_elemento, (torch.Tensor, np.ndarray)):
                print(f"  È un tensore/array diretto con shape: {list(primo_elemento.shape)}")
            else:
                print(f"  Valore: {primo_elemento}")

        else:
            print(f"Il file contiene un oggetto di tipo: {type(dati)}")

    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")

# --- COME USARLO ---

percorso_esempio = Path("/content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/cosplace_resnet18_512/svox_sun/default_metric/2026-05-15_15-37-26/preds_loftr/001.torch")
esplora_lista_torch(percorso_esempio)

🔍 Ispezionando il file: 001.torch
--------------------------------------------------
📦 Tipo di contenitore: LISTA con 20 elementi.

Scartiamo l'involucro. Il primo elemento (Indice 0) è di tipo: dict

Trovato il Dizionario! Contiene 10 chiavi:
  🔑 num_inliers     | Tipo: int           | Valore: 71
  🔑 H               | Tipo: ndarray       | Shape: [3, 3]
  🔑 all_kpts0       | Tipo: NoneType      | Valore: None
  🔑 all_kpts1       | Tipo: NoneType      | Valore: None
  🔑 all_desc0       | Tipo: NoneType      | Valore: None
  🔑 all_desc1       | Tipo: NoneType      | Valore: None
  🔑 matched_kpts0   | Tipo: ndarray       | Shape: [263, 2]
  🔑 matched_kpts1   | Tipo: ndarray       | Shape: [263, 2]
  🔑 inlier_kpts0    | Tipo: ndarray       | Shape: [71, 2]
  🔑 inlier_kpts1    | Tipo: ndarray       | Shape: [71, 2]


estrazione tempi retrieval


In [ ]:
!python extract_retrieval_times.py

Carico: /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/cosplace_resnet18_512/svox_sun/default_metric/2026-05-15_15-37-26/z_data.torch
  n_queries=854  t_ext=11.619s  t_ret=0.275s  t_tot=11.894s  (13.927 ms/query)
Carico: /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/cosplace_resnet18_512/svox_night/default_metric/2026-05-16_14-16-17/z_data.torch
  n_queries=823  t_ext=10.9798s  t_ret=0.2687s  t_tot=11.2486s  (13.668 ms/query)
Carico: /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/cosplace_resnet18_512/sf_xs/default_metric/2026-05-15_17-07-16/z_data.torch
  n_queries=1000  t_ext=14.1961s  t_ret=0.3275s  t_tot=14.5235s  (14.524 ms/query)
Carico: /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/cosplace_resnet18_512/tokyo_xs/default_metric/2026-05-15_16-49-39/z_data.torch
  n_queries=315  t_ext=6.7349s  t_ret=0.0696s  t_tot=6.8045s  (21.602 ms/query)
Carico: /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/m

analisi file stats complessive post image matching

In [ ]:
import torch
from pathlib import Path
import warnings

# Ignora il warning di sicurezza per il caricamento di oggetti non-tensoriali
warnings.filterwarnings("ignore", category=FutureWarning)

def leggi_statistiche_matching(percorso_file: str):
    file_path = Path(percorso_file)

    if not file_path.exists():
        print(f"❌ Errore: Il file '{file_path}' non esiste.")
        print("Assicurati di aver inserito il percorso corretto.")
        return

    try:
        # Caricamento del file
        stats = torch.load(file_path, map_location='cpu', weights_only=False)

        print(f"📊 Report Statistiche: {file_path.name}")
        print("=" * 50)

        # Stampa formattata se il contenuto è un dizionario
        if isinstance(stats, dict):
            for chiave, valore in stats.items():
                # Formattazione per allineare le chiavi
                print(f"🔹 {chiave:<30}: {valore}")
        else:
            print(f"⚠️ Il file non contiene un dizionario, ma un oggetto di tipo: {type(stats)}")
            print(stats)

        print("=" * 50)

    except Exception as e:
        print(f"❌ Si è verificato un errore durante la lettura: {e}")

if __name__ == "__main__":
    percorso = "/content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/netvlad_vgg16_4096/tokyo_xs/default_metric/2026-05-15_17-11-23/preds_loftr/matching_stats.torch"

    leggi_statistiche_matching(percorso)

📊 Report Statistiche: matching_stats.torch
🔹 matcher_name                  : loftr
🔹 num_queries_processed         : 315
🔹 num_preds_per_query           : 20
🔹 img_size                      : 512
🔹 mean_matching_time_per_query  : 3.9364248573906293
🔹 total_matching_time           : 1239.9738300780482
🔹 source_preds_dir              : /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/netvlad_vgg16_4096/tokyo_xs/default_metric/2026-05-15_17-11-23/preds


NUOVA SEZIONE


retrieval + image matching su datasets di train (svox sun/svox night train) e di validation (sf_xs val)

In [ ]:
!python VPR-methods-evaluation/main.py \
--num_workers 8 \
--batch_size 16 \
--log_dir "/content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/netvlad_vgg16_4096/svox_night_train/default_metric" \
--method="netvlad" \
--backbone="VGG16" \
--descriptors_dimension=4096 \
--distance_metric=l2 \
--image_size 512 512 \
--database_folder '/content/datasets/svox/images/train/gallery' \
--queries_folder '/content/datasets/svox/images/train/queries_night' \
--num_preds_to_save 2 \
--recall_values 1 2 \
--save_for_uncertainty

In [ ]:
!python match_queries_preds.py \
  --preds-dir "/content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/megaloc_resnet50_4096/svox_night_train/default_metric/2026-06-04_09-48-08/preds" \
  --matcher "superglue" \
  --device "cuda" \
  --num-preds 1

costruzione tabelle feature per regressore logistico

In [ ]:
!python /content/drive/MyDrive/Progetto_Machine_Learning/build_feature_csv_6_1_TEST.py


=== megaloc_resnet50_4096 | sf_xs | superpoint-lg ===
    run:      /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/megaloc_resnet50_4096/sf_xs/default_metric/2026-05-15_17-57-06
    OK  /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/megaloc_resnet50_4096/sf_xs/default_metric/2026-05-15_17-57-06/z_data.torch
    OK  /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/megaloc_resnet50_4096/sf_xs/default_metric/2026-05-15_17-57-06/preds
    OK  /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/megaloc_resnet50_4096/sf_xs/default_metric/2026-05-15_17-57-06/preds_superpoint-lg
    query: 1000 | match mancanti: 0 | top1_correct medio: 0.865
    -> salvato /content/drive/MyDrive/Progetto_Machine_Learning/feature_csv_6_1/test/features_megaloc_sf_xs_test_superpoint-lg.csv

=== megaloc_resnet50_4096 | tokyo_xs | superpoint-lg ===
    run:      /content/drive/MyDrive/Progetto_Machine_Learning/logs/retrieval/megaloc_resnet50_4096/tokyo_x

REGRESSORE logistico + CURVE ROC

In [ ]:
!python /content/drive/MyDrive/Progetto_Machine_Learning/logreg_6_1_alltests.py

PDF ROC (4/pagina): /content/drive/MyDrive/Progetto_Machine_Learning/results_6_1_alltests/roc_per_pagina_4.pdf

=== ROC-AUC per (caso, dataset, modello) ===
model                             baseline_gap  baseline_num_inliers  completo  proposta_clustered
case                  dataset                                                                     
megaloc_superglue     sf_xs              0.797                 0.944     0.962               0.852
                      svox_night         0.803                 0.919     0.942               0.914
                      svox_sun           0.736                 0.849     0.857               0.806
                      tokyo_xs           0.839                 0.961     0.965               0.900
megaloc_superpoint-lg sf_xs              0.797                 0.961     0.972               0.951
                      svox_night         0.803                 0.921     0.919               0.893
                      svox_sun           0.736     

applicazione soglia threshold

In [ ]:
!python /content/drive/MyDrive/Progetto_Machine_Learning/adaptive_recall_val_threshold.py


--- megaloc_superpoint-lg ---
    C=0.1  val_AUC=0.9713  val_thr=0.7013  val_F1=0.9891
    svox_sun : ret=96.4  full=96.0  adaptive=96.4  re-ranked=1.8%  skipped=98.2%
    svox_night: ret=95.7  full=90.3  adaptive=94.8  re-ranked=5.2%  skipped=94.8%
    sf_xs    : ret=86.5  full=86.6  adaptive=87.1  re-ranked=6.4%  skipped=93.6%
    tokyo_xs : ret=94.6  full=94.3  adaptive=94.6  re-ranked=1.3%  skipped=98.7%

--- megaloc_superglue ---
    C=0.1  val_AUC=0.9689  val_thr=0.6460  val_F1=0.9888
    svox_sun : ret=96.4  full=95.6  adaptive=96.4  re-ranked=1.1%  skipped=98.9%
    svox_night: ret=95.7  full=92.3  adaptive=96.0  re-ranked=3.2%  skipped=96.8%
    sf_xs    : ret=86.5  full=85.2  adaptive=86.6  re-ranked=3.8%  skipped=96.2%
    tokyo_xs : ret=94.6  full=92.4  adaptive=94.6  re-ranked=0.3%  skipped=99.7%

--- netvlad_superpoint-lg ---
    C=0.1  val_AUC=0.9745  val_thr=0.4249  val_F1=0.9508
    svox_sun : ret=37.1  full=65.2  adaptive=64.4  re-ranked=63.2%  skipped=36.8%
    svox

costruzione tabella con R@1 post full rerank per poter costruire poi la curva

In [ ]:
!python /content/drive/MyDrive/Progetto_Machine_Learning/build_reranked_correct_loop.py

=== Build reranked_correct da .torch ===

--- megaloc × superpoint-lg × sun ---
  cache già presente: /content/torch_cache/megaloc_sun_superpoint-lg
  OK megaloc  sun       superpoint-lg →  854 query  R@1=0.960  errori=0
  → debug: reranked_debug_megaloc_sun_superpoint-lg.csv

=== Aggiorno consolidato ===
Righe precedenti mantenute: 11114
Salvato: /content/drive/MyDrive/Progetto_Machine_Learning/feature_csv_6_1/test/rerank_eval_consolidato.csv  (11968 righe totali)

dataset                   night  sf_xs       sun  tokyo_xs
method  matcher                                           
megaloc superglue      0.923451  0.852  0.955504  0.923810
        superpoint-lg  0.902795  0.866  0.960187  0.942857
netvlad superglue      0.247874  0.608  0.642857  0.669841
        superpoint-lg  0.243013  0.623  0.652225  0.688889


costruzione curve adaptive reranking su dataset di test caricando i coefficienti dei regressori per ogni coppia metodo x matcher

In [ ]:
!python adaptive_reranking_curve.py


=== Carico coefficienti da /content/drive/MyDrive/Progetto_Machine_Learning/coefficients.csv ===

=== Calcolo curve ===

--- megaloc × superpoint-lg ---
  sun      : retrieval=0.964  full=0.960  gain=-0.004  breakeven=39%  saving=61%  threshold_P=0.9973
  night    : retrieval=0.957  full=0.903  gain=-0.055  breakeven=0%  saving=100%  threshold_P=0.3117
  sf_xs    : retrieval=0.865  full=0.866  gain=+0.001  breakeven=1%  saving=99%  threshold_P=0.4754
  tokyo_xs : retrieval=0.946  full=0.943  gain=-0.003  breakeven=0%  saving=100%  threshold_P=0.4928

--- megaloc × superglue ---
  sun      : retrieval=0.964  full=0.956  gain=-0.008  breakeven=56%  saving=44%  threshold_P=0.9991
  night    : retrieval=0.957  full=0.923  gain=-0.034  breakeven=0%  saving=100%  threshold_P=0.3489
  sf_xs    : retrieval=0.865  full=0.852  gain=-0.013  breakeven=0%  saving=100%  threshold_P=0.4200
  tokyo_xs : retrieval=0.946  full=0.924  gain=-0.022  breakeven=0%  saving=100%  threshold_P=0.5830

--- netvl

In [2]:
!du -sh /content/drive/MyDrive/Progetto_Machine_Learning/*/

4.5M	/content/drive/MyDrive/Progetto_Machine_Learning/analisi_risultati_vpr/
4.5M	/content/drive/MyDrive/Progetto_Machine_Learning/analysis_outputs/
9.7G	/content/drive/MyDrive/Progetto_Machine_Learning/datasets/
8.4M	/content/drive/MyDrive/Progetto_Machine_Learning/feature_csv_6_1/
25G	/content/drive/MyDrive/Progetto_Machine_Learning/logs/
65M	/content/drive/MyDrive/Progetto_Machine_Learning/pip_cache/
1.4M	/content/drive/MyDrive/Progetto_Machine_Learning/results_6_1_alltests/
2.7G	/content/drive/MyDrive/Progetto_Machine_Learning/Visual-Place-Recognition-Project/
